# Section Estimators with `Scikit-Learn`

The task is to estimate the properties of a particular cross section.

In [ ]:
from typing import Optional

import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.base import BaseEstimator
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
)
from utils import CANONICAL_SCORE_NAME, CROSS_SECTION_PARAMETERS, print_system_info
from utils.ml import canonical_regression_score
import numpy as np
import mlflow
import json

print_system_info()

In [28]:
config_file_path = "config_rhs.json"
data_file_path = "data_rhs.csv"
mlflow_tracking_uri="sqlite:///mlflow.db"
mlflow_experiment_name=None
task = "section_estimation"

In [29]:
with open(config_file_path, "r") as f:
    config: dict = json.load(f)

In [ ]:
# Load section data
section_data = config["section"]

section_type = config["section"]["type"]
print(f"Section type from config: {section_type}")

section_variables = []
for p in section_data["params"].keys():
    if section_data["params"][p]["variable"]:
        section_variables.append(p)

predictor_columns = section_variables

target_columns = CROSS_SECTION_PARAMETERS

if not mlflow_experiment_name:
    mlflow_experiment_name = f"{task}__{section_type}"
    
mlflow.set_tracking_uri(mlflow_tracking_uri)
experiment = mlflow.set_experiment(mlflow_experiment_name)
mlflow.set_experiment_tag("section_type", section_type)
mlflow.set_experiment_tag("task", task)
mlflow.sklearn.autolog(log_models=False)

Section type from config: rectangular_hollow_section


## Prepare data

In [31]:
df = pd.read_csv(data_file_path)
df = df.dropna()
df = df.drop_duplicates(subset="section_param_id")
assert df.isnull().sum().max() == 0, "DataFrame still contains NaN values after dropping."
print("Number of rows after dropping NaNs:", len(df))
df.head(5)

Number of rows after dropping NaNs: 186


,d,b,t,r_out,n_r,n,mxx,myy,vx,vy,mzz,area,ixx,iyy,ixy,g_eff,utilization,section_param_id,section_type
0,292.940338,184.123599,16.722934,17.628862,4,-688877.841108,-1.312324e+07,2.412614e+06,-115170.575087,13682.531592,1.153415e+07,14527.235375,3.200690e+13,1.520191e+13,-0.265625,76923.076923,0.322426,0,rectangular_hollow_section
20,261.663034,194.143300,14.020467,9.775481,4,-494041.944866,-1.103744e+07,-1.040838e+07,-3421.483883,249276.708267,1.397299e+07,11899.381255,2.235461e+13,1.389697e+13,0.101562,76923.076923,0.290937,1,rectangular_hollow_section
40,290.075336,186.244780,3.650917,8.201333,4,233467.851888,-1.679999e+07,3.599557e+06,179159.912102,144302.129765,-9.333793e+06,3378.137930,8.136555e+12,4.149276e+12,-0.007812,76923.076923,0.856259,2,rectangular_hollow_section
80,187.477426,155.818205,10.721940,15.424910,4,-706993.621694,2.099932e+07,-9.300146e+05,-179324.334873,-147967.487206,8.658481e+06,6685.940561,6.505753e+12,4.863943e+12,-0.023438,76923.076923,0.556805,4,rectangular_hollow_section
100,101.813572,109.591089,13.930502,10.252011,4,-410045.478994,-9.123781e+06,-2.871402e+07,125610.178319,-131835.604272,-1.815835e+07,5008.606822,1.327432e+12,1.502990e+12,0.010254,76923.076923,1.025403,5,rectangular_hollow_section


In [32]:
X = df[predictor_columns]
y = df[target_columns]
X.shape, y.shape

((186, 4), (186, 5))

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (124, 4)
Testing data shape: (62, 4)


## Train models

In [ ]:
def train_model(
    model: BaseEstimator, 
    params_grid: Optional[dict] = None, 
    model_name: Optional[str] = None
) -> BaseEstimator:
    run = mlflow.start_run(run_name=model_name or model.__class__.__name__)  # --- start main run
    run_id = run.info.run_id
    try:
        # ----- tags & baseline params
        mlflow.set_tag("model_class", model.__class__.__name__)
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("stage", "baseline" if not params_grid else "baseline+search")
        mlflow.set_tag("task", task)
        mlflow.set_tag("section_type", section_type)
        mlflow.set_tag("library", "sklearn")

        # ----- fit on train
        model.fit(X_train, y_train.values)
        
        # define cross-validation strategy and log parameters
        kf_params = {"n_splits": 6, "random_state": 42, "shuffle": True}
        kf = KFold(**kf_params)
        logged_kf_params = {f"kf__{k}": v for k, v in kf_params.items()}
        mlflow.log_params(logged_kf_params)

        # ----- hyperparameter search (optional)
        if params_grid:
            mlflow.log_dict(params_grid, "param_grid.json")
            mlflow.start_run(run_name=f"{model_name} - RandomizedSearchCV", nested=True)  #-- start nested run
            try:
                cv = RandomizedSearchCV(
                    estimator=model,
                    param_distributions=params_grid,
                    cv=kf,
                    n_iter=10,
                    n_jobs=-1,
                    random_state=42,
                    refit=True,
                )
                cv.fit(X_train, y_train.values)
                model = cv.best_estimator_
            finally:
                mlflow.end_run()  # --- end nested run
        
        # log the final model
        mlflow.sklearn.log_model(model, name=model_name, input_example=X_train.iloc[:5])
        
        # log cv scores
        cv_scores = cross_val_score(model, X_train, y_train.values, cv=kf)
        mlflow.log_metric("train_cv_score_mean", float(np.mean(cv_scores)))
        mlflow.log_metric("train_cv_score_std", float(np.std(cv_scores)))
        
        # log regression metrics on test set
        y_true = y_test.values
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)
        medae = median_absolute_error(y_true, y_pred)
        metrics = {
            "test_mae": mae,
            "test_mse": mse,
            "test_rmse": rmse,
            "test_r2": r2,
            "test_mape": mape,
            "test_medae": medae,
        }
        mlflow.log_metrics(metrics)
        
        # log canonical score - the higher the better
        canonical_score = canonical_regression_score(y_true, y_pred)
        mlflow.log_metric(CANONICAL_SCORE_NAME, canonical_score)
        
        return run_id, model

    finally:
        mlflow.end_run()  # --- end main run
        print("MLflow run ended.")

### Baseline model

In [36]:
model_name = "Linear Regression Baseline"
steps = [
    ("scaling", StandardScaler()),
    ("regression", LinearRegression())
]
pipeline = Pipeline(steps)

run_id, model = train_model(pipeline, model_name=model_name)

MLflow run ended.


### Tuned models

In [37]:
model_name = "Ridge Regression"
steps = [
    ("scaling", StandardScaler()),
    ("regression", Ridge(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/12 15:24:34 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.


MLflow run ended.


In [38]:
model_name = "Lasso Regression"
steps = [
    ("scaling", StandardScaler()),
    ("regression", Lasso(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.795e+26, tolerance: 6.072e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to i

MLflow run ended.


/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.795e+26, tolerance: 6.072e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.397e+26, tolerance: 6.140e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/s

In [39]:
model_name = "Lasso Regression Poly3"
steps = [
    ("feature_eng", PolynomialFeatures(3, include_bias=False)),
    ("scaling", StandardScaler()),
    ("regression", Lasso(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.039e+24, tolerance: 7.481e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.992e+24, tolerance: 6.856e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/s

MLflow run ended.


/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.400e+24, tolerance: 4.902e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.116e+24, tolerance: 4.436e+23
  model = cd_fast.enet_coordinate_descent(
/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/s

In [40]:
model_name = "Ridge Regression Poly3"
steps = [
    ("feature_eng", PolynomialFeatures(3, include_bias=False)),
    ("scaling", StandardScaler()),
    ("regression", Ridge(alpha=0.1))
]
pipeline = Pipeline(steps)
params_grid = {"regression__alpha": np.arange(0.0001, 1, 10)}

run_id, model = train_model(pipeline, params_grid=params_grid, model_name=model_name)

/Users/baloghbence/Library/Caches/pypoetry/virtualenvs/sigmaepsilon-solid-fourier-2jj81J8s-py3.12/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2025/11/12 15:24:40 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.


MLflow run ended.
